In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. CARGA DE DATOS
print("--- Cargando Matriz RFM ---")
rfm = pd.read_csv('rfm_data.csv', index_col='CustomerID')

# 2. PREPROCESAMIENTO DE DATOS (CRÍTICO EN IA)
# a) Transformación Logarítmica:
# Los datos financieros suelen tener sesgo (skewness). Usamos log para "suavizar" los picos
# y que las "Ballenas" (VIPs extremos) no rompan el modelo.
rfm_log = np.log1p(rfm)

# b) Estandarización (StandardScaler):
# Ponemos todo en la misma escala (Media 0, Desviación Estándar 1).
# Así, 10 días de recencia pesan lo mismo que $100 dólares.
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# 3. ENTRENAMIENTO DEL MODELO (K-MEANS)
print("--- Ejecutando K-Means Clustering ---")
# Definimos 4 clusters (Arquetipos estándar: VIP, Leales, Nuevos, Perdidos)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(rfm_scaled)

# 4. ASIGNACIÓN
# Le pegamos la etiqueta (0, 1, 2, 3) a cada cliente en el dataframe original
rfm['Cluster'] = clusters

# 5. INTERPRETACIÓN DE NEGOCIO
# Agrupamos por cluster para ver los promedios y entender "quién es quién"
perfil_clusters = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(1)

print("\n--- Perfiles Detectados por la IA ---")
print(perfil_clusters)

# 6. GUARDADO
rfm.to_csv('rfm_labeled.csv')
print("\n✅ Archivo 'rfm_labeled.csv' guardado. Listo para visualizar.")

--- Cargando Matriz RFM ---
--- Ejecutando K-Means Clustering ---

--- Perfiles Detectados por la IA ---
        Recency Frequency Monetary      
           mean      mean     mean count
Cluster                                 
0          81.8       5.3    176.8   177
1          17.3      12.6    382.4   198
2         147.6       1.6     26.2   136
3           7.7      45.6   1425.2   139

✅ Archivo 'rfm_labeled.csv' guardado. Listo para visualizar.
